In [9]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数 - 完美对齐 BigAlpha 2026 官方赛制规范
    核心因子：中证 1000 纯净 10 日动量反转因子 (-f3)
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import structlog

    logger = structlog.get_logger()

    # 从映射里取出本阶段实际的分钟物理表名
    bar1m_table = datasources["bar1m"]

    # 考虑到要计算 10 日时序滚动，向前多取 30 天作为数据缓冲期
    LOOKBACK_DAYS = 30
    query_start_date = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d 00:00:00")
    query_end_date = pd.to_datetime(end_date).strftime("%Y-%m-%d 23:59:59")

    t0 = time.time()

    bar1m_table = datasources["bar1m"]
    financial_table = datasources["financial"]

    sql = f"""
    WITH cte_financial AS (
        SELECT 
            date::DATE::DATETIME AS date,
            instrument,
            -- 【修复】：使用官方真实存在的 net_profit_deducted (扣非净利润) 
            -- 游资博弈中，扣非净利润越高，说明基本面越扎实，非垃圾壳股
            ARG_MAX(net_profit_deducted, date) AS financial_metric
        FROM {financial_table}
        GROUP BY date::DATE, instrument
    ),
    cte_micro_bar AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            -- 1. 提取日尾收盘价
            ARG_MAX(close, date) AS close,
            -- 2. 聚合日内 3 档买方力量（低位承接盘厚度）
            AVG(bid_volume1 + bid_volume2 + bid_volume3) AS avg_bid_depth,
            -- 3. 聚合日内 3 档卖方力量
            AVG(ask_volume1 + ask_volume2 + ask_volume3) AS avg_ask_depth
        FROM {bar1m_table}
        WHERE ask_price1 > 0 AND bid_price1 > 0
        GROUP BY date::DATE, instrument
    ),
    cte_combined_base AS (
        SELECT
            date,
            instrument,
            close,
            -- 计算盘口买卖失衡度（买盘越厚，说明低位承接力越强）
            (avg_bid_depth - avg_ask_depth) / (avg_bid_depth + avg_ask_depth + 1e-5) AS orderbook_imbalance,
            -- 融合财务盈利能力
            COALESCE(financial_metric, 0) AS f_metric
        FROM cte_micro_bar
        LEFT JOIN cte_financial USING (date, instrument)
    ),
    cte_final_factor AS (
        SELECT
            date,
            instrument,
            -- 【终极公式】：10日超跌反转 * 横截面买盘支撑度排名 * 横截面扣非净利润排名
            -1.0 * c_pct_rank(
                ((close - m_lag(close, 10)) / (m_lag(close, 10) + 1e-5)) 
                * c_pct_rank(orderbook_imbalance)
                * c_pct_rank(f_metric)
            ) as factor
        FROM cte_combined_base
        WHERE date BETWEEN '{query_start_date}' AND '{query_end_date}'
    )
    SELECT
        date,
        instrument::STRING AS instrument,
        factor
    FROM cte_final_factor
    WHERE factor IS NOT NULL
    ORDER BY date, instrument
    """

    df = dai.query(sql, full_db_scan=True, compression=True).df()
    df['date'] = pd.to_datetime(df['date'])

    df = df[(df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))]
    logger.info(f"📈 因子截面打分完毕，耗时: {round(time.time()-t0, 2)}秒，正在对齐中证 1000 股票池...")

    # ===== 精准对齐官方赛制指定的股票池（中证 1000 历史成分股大底座） =====
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    # 实施 inner join 强对齐，保证覆盖度 100% 通过
    result = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])
    result = result.dropna(subset=['factor']).reset_index(drop=True)
    
    logger.info(f"🏁 官方合规性检验通过！最终交付样本行数: {len(result)}")
    return result[['date', 'instrument', 'factor']]


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    
    # 完美复刻评测程序的自测注入环境
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m', 
        'financial': 'bigalpha_2026_financial'
    }
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 运行主流程
    factor_data = main(datasources, start_date, end_date)

    # 读取赛事专用官方基准因子库进行多维评估
    logger.info(f"提取 bigalpha_2026_factorlib 进行多维正交回归评估...")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data, factor_pool=factor_pool, process_pools=False, show=True
    )

[2026-07-04 21:53:22] [info     ] 📈 因子截面打分完毕，耗时: 15.93秒，正在对齐中证 1000 股票池...
[2026-07-04 21:53:23] [info     ] 🏁 官方合规性检验通过！最终交付样本行数: 241764
[2026-07-04 21:53:23] [info     ] 提取 bigalpha_2026_factorlib 进行多维正交回归评估...
[2026-07-04 21:53:23] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)
[2026-07-04 21:53:26] [info     ] bigalpha_eval.v4 开始运行 ..
[2026-07-04 21:53:29] [warning  ] 未传入官方评估窗口 start_date/end_date，回退到数据自身范围（仅建议本地调试时使用）
[2026-07-04 21:53:33] [info     ] 对齐中证1000历史成分后，官方评估窗口: 2024-01-02 至 2024-12-31
[2026-07-04 21:53:33] [info     ] ========== 数据检查 ==========
[2026-07-04 21:53:33] [info     ] 通过：列名检查（date/instrument + 至少 1 个因子列） factor_cols=['factor', 'close', 'volume', 'amount', 'turn', 'change_ratio', 'daily_return', 'momentum_5', 'reversal_5', 'volatility_5', 'total_market_cap', 'float_market_cap', 'pe_ttm', 'pb', 'ps_ttm', 'sma_20', 'ema_20', 'macd_diff_12_26_9', 'macd_dea_12_26_9', 'macd_hist_12_26_9', 'rsi_12', 'kdj_k_9_3_3', 'kdj_d_9_3